In [ ]:
from utils import train_categorical_model, prepare_datasets,validate_results
from rcf_model import score_rcf_streaming
from scoring import optimize_fusion_weights, risk_fusion_scorer

In [ ]:
file_path = r"UNSW-NB15\unsw_train.csv"
X_train_cat, X_train_num, y_train, cat_cols = prepare_datasets(file_path)
catboost_model = train_categorical_model(X_train_cat, y_train, cat_cols)
print("\nGenerating raw cat scores for optimization...")
cat_scores_raw = catboost_model.predict_proba(X_train_cat)[:, 1]

print("Scoring numerical data through custom RCF (Streaming point-by-point)...")
rcf_scores_norm = score_rcf_streaming(X_train_num)

Loading and transforming data from UNSW-NB15\unsw_train.csv...
Final Categorical features: 6
Numerical features compressed from 36 down to 20 Principal Components

Training CatBoost (with High Regularization)...

Generating raw cat scores for optimization...
Scoring numerical data through custom RCF (Streaming point-by-point)...


RRCF Scoring Progress: 100%|██████████| 82332/82332 [12:19<00:00, 111.33it/s]



Optimizing Fusion Hyperparameter...
Optimal Categorical Weight: 0.80
Optimal Numerical (RCF) Weight: 0.20
Resulting F1-Score: 0.8520

Pipeline Output for first 5 events:
Event 1 | Cat Risk: 0.46 | Num Anomaly: 0.00 | FUSED RISK: 0.37
Event 2 | Cat Risk: 0.46 | Num Anomaly: 0.00 | FUSED RISK: 0.37
Event 3 | Cat Risk: 0.46 | Num Anomaly: 0.00 | FUSED RISK: 0.37
Event 4 | Cat Risk: 0.46 | Num Anomaly: 0.00 | FUSED RISK: 0.37
Event 5 | Cat Risk: 0.46 | Num Anomaly: 0.00 | FUSED RISK: 0.37


In [27]:
optimal_cat_weight, optimal_threshold = optimize_fusion_weights(y_train, cat_scores_raw, rcf_scores_norm,COST_FN=10, COST_FP=2)
final_risk = risk_fusion_scorer(cat_scores_raw, rcf_scores_norm, cat_weight=optimal_cat_weight)
validate_results(y_train, final_risk, catboost_model, cat_cols,optimal_threshold)


Running Cost-Sensitive 2D Optimization (Weight + Threshold)...
Optimal Categorical Weight: 0.80
Optimal Numerical (RCF) Weight: 0.20
Optimal Alert Threshold: 0.30
Lowest Operational Penalty Score: 32004

--- PIPELINE VALIDATION ---

1. Classification Report (Operational Impact):
              precision    recall  f1-score   support

  Normal (0)       1.00      0.58      0.73     37000
  Attack (1)       0.74      1.00      0.85     45332

    accuracy                           0.81     82332
   macro avg       0.87      0.79      0.79     82332
weighted avg       0.86      0.81      0.80     82332


2. Confusion Matrix:
True Negatives (Correctly Allowed):  21378
False Positives (Unjustified Blocks): 15622  <-- Alert Fatigue Risk
False Negatives (Missed Attacks):     76  <-- Security Breach Risk
True Positives (Correctly Blocked):   45256

3. Top 5 Drivers of Categorical Risk:
 - ct_state_ttl: 40.28% influence
 - service: 27.34% influence
 - proto: 26.62% influence
 - is_sm_ips_ports: